# 01 Python and Matrix Foundations

## Problem

为什么在进入 tokenizer、embedding、attention 之前，必须先把向量、矩阵、batch、softmax、mask 这些基础概念真正讲清楚？

## Dependencies

会基本 Python 语法即可。即使你对线性代数不熟，也可以沿着这里的最小例子往下看。


## Goals

- 理解向量、矩阵、batch、sequence length 在 LLM 里分别表示什么
- 理解矩阵乘法为什么可以看成“特征重组”
- 理解 softmax 为什么能把一组分数变成概率分布
- 理解 causal mask 为什么能阻止模型偷看未来 token
- 建立一个后面会反复使用的直觉：现代 LLM 处理的核心对象通常是 `(batch, seq_len, hidden_dim)` 这样的张量

## Scope and Approach

这一节不追求把线性代数当课程完整讲一遍，而是只抓住后面最会反复出现的那几个概念：shape、矩阵乘法、softmax、mask。目标不是让读者会推所有公式，而是让读者在看到后面任何一段 attention 代码时，至少知道数据长什么样、每一步在做什么、为什么要这么做。


## Why this section matters

很多人学 Transformer 时，一上来就看 `Q @ K.T`、`softmax`、`mask`，结果每个单词都眼熟，但连起来还是糊。根本原因通常不是 attention 本身太玄，而是更底层的“张量直觉”没有建立起来。

先把几个最常见的词讲清楚：

- **向量**：一条样本或一个位置的特征列表
- **矩阵**：很多个向量排成一张表
- **batch**：一次并行处理多少条样本
- **sequence length**：一条文本里有多少个位置或 token
- **hidden dimension**：每个位置当前用多少维数字来表示自己

如果把这些概念只当数学名词，会很抽象；如果把它们理解成“内存里排成什么形状的数据”，很多看起来复杂的式子会突然变简单。


## First mental model: what a model actually sees

假设我们现在还不谈 tokenization，只假设模型已经拿到了一些数值特征。那模型看到的通常不是一句话，而是一块三维数据：

- 第一维：batch，表示几条样本一起算
- 第二维：sequence length，表示一条样本里有几个位置
- 第三维：hidden dimension，表示每个位置有多少个特征值

后面几乎所有核心模块，都是在这个三维张量上做事：

- 有的模块改最后一维，也就是重组特征
- 有的模块在第二维上建立位置之间的关系
- 有的模块在不改变 shape 的情况下调整数值分布


In [ ]:
import numpy as np

# 让 numpy 打印得更紧凑一些，方便观察数值
np.set_printoptions(precision=3, suppress=True)

# x 的 shape = (2, 3, 4)
# 2 表示 batch size = 2，也就是这里一次并行处理 2 条样本
# 3 表示 sequence length = 3，也就是每条样本有 3 个位置
# 4 表示 hidden dimension = 4，也就是每个位置用 4 个数来表示
x = np.array([
    [
        [1.0, 0.0, 1.0, 0.0],
        [0.5, 1.0, 0.0, 0.0],
        [0.0, 1.0, 1.0, 0.5],
    ],
    [
        [0.2, 0.3, 0.7, 0.8],
        [1.0, 1.0, 0.0, 0.0],
        [0.4, 0.2, 0.1, 0.9],
    ],
])

print('x.shape =', x.shape)
print('第一条样本的 shape =', x[0].shape)
print('第一条样本的第一个位置 =', x[0, 0])
print('第二条样本的第三个位置 =', x[1, 2])


## Matrix multiplication as feature mixing

很多人第一次看到矩阵乘法时，只会把它理解成一种机械运算。对后面的模型实现来说，更有用的理解是：

**矩阵乘法本质上是在重新组合特征。**

比如一个位置原来有 4 个特征，乘上一个 `4 x 3` 的权重矩阵之后，就会变成 3 个新特征。这里真正发生的事情是：

- 原来的 4 个特征按不同权重重新混合
- 混合后形成新的表示空间
- 后面的网络就不再直接看旧特征，而是看新特征

这也是为什么线性层虽然叫“线性”，但仍然非常重要：它决定模型用什么视角重新看输入。


In [ ]:
# W 的 shape = (4, 3)
# 这表示：把原来每个位置的 4 维特征，映射成新的 3 维特征
W = np.array([
    [1.0, 0.0, 0.5],
    [0.0, 1.0, 0.5],
    [0.5, 0.5, 1.0],
    [1.0, -0.5, 0.0],
])

# x 的最后一维是 4，正好和 W 的第一维 4 对齐，所以可以相乘
# 结果 projected 的 shape 会变成 (2, 3, 3)
# 也就是 batch 和 sequence length 不变，但 hidden dimension 从 4 变成 3
projected = x @ W

print('W.shape =', W.shape)
print('projected.shape =', projected.shape)
print('第一条样本投影后的表示 =')
print(projected[0])


## Why softmax shows up everywhere

模型里经常先算出一堆分数，但分数本身不适合直接当权重用。原因很简单：

- 分数可能有负数
- 分数的总和不受控制
- 分数彼此之间不容易解释成“谁更重要”

softmax 的作用就是把一组任意实数，变成一组：

- 都大于 0 的数
- 总和等于 1
- 可以解释成概率或注意力权重

所以你后面在 attention 里看到 softmax，可以先把它理解成一句话：

**把一行相关性分数，变成一行“该关注谁、关注多少”的权重。**


In [ ]:
def softmax(logits):
    # 先减去最大值，是经典的数值稳定技巧
    # 这样不会改变 softmax 结果，但能避免 exp 太大
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)

scores = np.array([2.0, 1.0, 0.1])
probs = softmax(scores)

print('scores =', scores)
print('softmax(scores) =', probs)
print('概率和 =', probs.sum())


## Why masking matters

在 decoder-only 语言模型里，第 `t` 个位置预测时，不能偷看未来位置。比如：

- 第一个词不能看第二个词
- 第二个词不能看第三个词
- 当前位置只能看自己和它之前的位置

mask 的作用就是把“不允许看”的位置打掉。最常见的做法不是把它们删掉，而是：

- 给这些位置加一个极小的负数，比如 `-1e9`
- 这样经过 softmax 之后，这些位置的权重会接近 0

这就是 causal mask 的核心机制。


In [ ]:
# 下面构造一个长度为 4 的 causal mask
# 对角线以上的位置表示“未来”，这些位置要被屏蔽掉
seq_len = 4
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -1e9

print('causal_mask =')
print(causal_mask)

# 假设这是某个位置对所有位置的原始打分矩阵
raw_scores = np.array([
    [1.2, 0.5, 0.1, 0.8],
    [0.4, 1.1, 0.3, 0.2],
    [0.6, 0.7, 1.4, 0.9],
    [0.2, 0.1, 0.3, 1.5],
])

# 把 mask 加上去后，未来位置会变成极小值
masked_scores = raw_scores + causal_mask
masked_probs = softmax(masked_scores)

print('raw_scores =')
print(raw_scores)
print('masked_scores =')
print(masked_scores)
print('masked_probs =')
print(masked_probs)


## Common mistakes

- 把 batch size 和 sequence length 混为一谈。前者是并行多少条样本，后者是一条样本里有多少个位置。
- 把矩阵乘法当成纯数学技巧，而没看到它其实是在做特征重组。
- 看到 softmax 就只记公式，不去理解它为什么适合做权重归一化。
- 以为 mask 是把元素“删除”了。更常见的实现其实是把它们的分数压到极小。

## Checkpoints

- 把 `x` 改成 `shape=(1, 5, 4)`，重新观察三维张量的含义。
- 把 `scores = [2.0, 1.0, 0.1]` 改成 `[20.0, 10.0, 1.0]`，观察 softmax 会变得多尖锐。
- 自己手写一个长度为 5 的 causal mask，并解释每一行允许看哪些位置。
- 用自己的话解释为什么后面的大部分模块都可以看成是在处理 `(batch, seq_len, hidden_dim)`。

## Summary

这一节真正要建立的，不是零散公式，而是一种工作直觉：现代 LLM 的大部分模块，本质上都在处理一个三维张量 `(batch, seq_len, hidden_dim)`。矩阵乘法负责重组特征，softmax 负责把分数变成权重，mask 负责限制哪些位置可以参与计算。把这三件事理解透，后面再看 tokenizer、embedding、attention，就不会一直像在背咒语。

## Next Module

下一节进入 tokenizer 和 BPE。那时你会看到，模型吃进去的第一口并不是“字符”或“单词”，而是更适合统计建模的 token。
